In [3]:
import importlib
import tiktoken

In [4]:
with open("wizard_of_oz.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    
print("Total number of character:", len(raw_text))
print(raw_text[:1000])

Total number of character: 252060
﻿The Project Gutenberg eBook of Dorothy and the Wizard in Oz
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using this eBook.

Title: Dorothy and the Wizard in Oz

Author: L. Frank Baum

Illustrator: John R. Neill


        
Release date: September 10, 2007 [eBook #22566]

Language: English

Other information and formats: www.gutenberg.org/ebooks/22566

Credits: Produced by Chris Curnow, Joseph Cooper, Janet Blenkinship
        and the Online Distributed Proofreading Team at
        http://www.pgdp.net


*** START OF THE PROJECT GUTENBERG EBOOK DOROTHY AND THE WIZARD IN OZ ***

In [5]:
tokenizer = tiktoken.get_encoding("gpt2")
enc_text=tokenizer.encode(raw_text)
print(len(enc_text))

65688


In [28]:
enc_sample=enc_text[100:]

In [29]:
context_size=4
x=enc_sample[:context_size]
y=enc_sample[1:context_size+1]
print(f"x:{x}")
print(f"y:     {y}")

x:[198, 5832, 481, 423]
y:     [5832, 481, 423, 284]


In [30]:
for i in range(1,context_size+1):
    context=enc_sample[:i]
    desired=enc_sample[i]
    print(context,"---->",desired)

[198] ----> 5832
[198, 5832] ----> 481
[198, 5832, 481] ----> 423
[198, 5832, 481, 423] ----> 284


In [31]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))


 ----> you

you ---->  will

you will ---->  have

you will have ---->  to


In [32]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [33]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [78]:
import torch
print("PyTorch version:", torch.__version__)
dataloader = create_dataloader_v1(
    raw_text, batch_size=2, max_length=4 , stride=2, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

PyTorch version: 2.7.1+cu118
[tensor([[  171,   119,   123,   464],
        [  123,   464,  4935, 20336]]), tensor([[  119,   123,   464,  4935],
        [  464,  4935, 20336, 46566]])]


In [79]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 4935, 20336, 46566,   286],
        [46566,   286, 40349,   290]]), tensor([[20336, 46566,   286, 40349],
        [  286, 40349,   290,   262]])]


In [81]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=2, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[  171,   119,   123,   464],
        [  123,   464,  4935, 20336],
        [ 4935, 20336, 46566,   286],
        [46566,   286, 40349,   290],
        [40349,   290,   262, 16884],
        [  262, 16884,   287, 18024],
        [  287, 18024,   198,   220],
        [  198,   220,   220,   220]])

Targets:
 tensor([[  119,   123,   464,  4935],
        [  464,  4935, 20336, 46566],
        [20336, 46566,   286, 40349],
        [  286, 40349,   290,   262],
        [  290,   262, 16884,   287],
        [16884,   287, 18024,   198],
        [18024,   198,   220,   220],
        [  220,   220,   220,   220]])
